# Ames Housing - Machine Learning Pipeline

## Project Objective

The objective of this project is to build a complete machine learning pipeline for house price prediction using the Ames Housing Dataset.

This notebook focuses on:

- Automated preprocessing
- Ordinal encoding
- One-hot encoding
- Numerical transformations
- Feature engineering
- Pipeline development
- Model evaluation

Problem Type:

- Regression

Target Variable:

- SalePrice

In [330]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,StandardScaler,PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

In [331]:
df = pd.read_csv("/content/train.csv")

In [332]:
df.shape

(1460, 81)

In [333]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [334]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [335]:
df.drop(['Id','Utilities','Street'],axis=1,inplace=True)

In [336]:
X = df.drop('SalePrice',axis = 1)
y = df['SalePrice']

# Train-Test Split

The dataset is divided into training and testing subsets before preprocessing.

This prevents data leakage and ensures unbiased model evaluation.

In [337]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size= 0.2,
    random_state= 42
)

In [338]:
X_train.shape

(1168, 77)

In [339]:
X_test.shape

(292, 77)

In [340]:
num_cols = X_train.select_dtypes(
    include = ['int64','float64']
).columns.tolist()

In [341]:
cat_cols =  X_train.select_dtypes(
    include = ['object','category']
).columns.tolist()

# Feature Categorization

Features are categorized into:

## Numerical Features

Continuous and discrete numerical variables.

## Ordinal Features

Categorical variables with natural ordering.

Examples:

- ExterQual
- KitchenQual
- GarageCond

## Nominal Features

Categorical variables without natural ordering.

Examples:

- Neighborhood
- GarageType
- SaleCondition

In [342]:
ordinal_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'HeatingQC',
    'KitchenQual',
    'FireplaceQu',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'GarageFinish',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'LotShape',
    'PavedDrive'
]

In [343]:
nominal_cols = [
    col
    for col in cat_cols
    if col not in ordinal_cols
]

In [344]:
quality_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']

# Ordinal Encoding Strategy

Several housing quality features contain ordered categories.

Examples:

Ex > Gd > TA > Fa > Po

Ordinal encoding preserves the ranking information present in these variables.

In [345]:
quality_pipeline = Pipeline([

    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),
    ( 'encoder',
        OrdinalEncoder(
            categories=[
                ['None','Po','Fa','TA','Gd','Ex']
            ] * len(quality_cols),
            handle_unknown='use_encoded_value',
            unknown_value=-1
            ))
])

In [346]:
bsmt_fin_cols = ['BsmtFinType1','BsmtFinType2']

In [347]:
bsmt_fin_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),
    ('encoder',OrdinalEncoder(
        categories=[['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ']]*2,
        handle_unknown='use_encoded_value',
        unknown_value = -1
    ))
])

In [348]:
garage_finish_cols = ['GarageFinish']

In [349]:
garage_finish_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),
    ('encoder',OrdinalEncoder(
        categories=[['None','Unf','RFn','Fin']],
        handle_unknown='use_encoded_value',
        unknown_value= -1
    ))
])

In [350]:
bsmt_exposure_cols = ['BsmtExposure']

In [351]:
bsmt_exposure_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),
    ('encoder',OrdinalEncoder(
        categories=[['None','No','Mn','Av','Gd']],
        handle_unknown='use_encoded_value',
        unknown_value= -1
    ))
])

In [352]:
lotshape_cols = ['LotShape']

In [353]:
lotshape_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),
    ('encoder',OrdinalEncoder(
        categories=[['IR3','IR2','IR1','Reg']],
        handle_unknown='use_encoded_value',
        unknown_value= -1
    ))
])

In [354]:
paved_drive_cols = ['PavedDrive']

In [355]:
paved_drive_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),
    ('encoder',OrdinalEncoder(
        categories=[['N','P','Y']],
        handle_unknown='use_encoded_value',
        unknown_value= -1
    ))
])

In [356]:
ordinal_preprocessor = ColumnTransformer([
    ('quality',quality_pipeline,quality_cols),
    ('garage_finish',garage_finish_pipeline,garage_finish_cols),
    ('bsmt_exposure', bsmt_exposure_pipeline,bsmt_exposure_cols),
    ('bsmt_fin', bsmt_fin_pipeline,bsmt_fin_cols),
    ('lotshape', lotshape_pipeline,lotshape_cols),
    ('paved_drive',paved_drive_pipeline,paved_drive_cols)

])

In [357]:
nominal_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='Missing')),
    ('encoder',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'))
])

# Numerical Preprocessing

The numerical pipeline performs:

1. Missing value imputation using median values.
2. Power transformation using Yeo-Johnson transformation.
3. Feature scaling using StandardScaler.

These steps help stabilize distributions and improve model performance.

In [358]:
num_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('transformer',PowerTransformer('yeo-johnson')),
    ('scaler',StandardScaler())
])

In [359]:
preprocessor = ColumnTransformer([
    ('ordinal_trf',ordinal_preprocessor,ordinal_cols),
    ('nominal_col',nominal_pipeline,nominal_cols),
    ('num_pipe',num_pipeline,num_cols)
])

# Baseline Model

The baseline model uses original dataset features without custom feature engineering.

Purpose:

- Establish benchmark performance
- Measure the effectiveness of engineered features

In [360]:
base_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
    ])

In [361]:
base_pipeline.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:188: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('ordinal_trf',
                                                  ColumnTransformer(transformers=[('quality',
                                                                                   Pipeline(steps=[('imputer',
                                                                                                    SimpleImputer(fill_value='None',
                                                                                                                  strategy='constant')),
                                                                                                   ('encoder',
                                                                                                    OrdinalEncoder(categories=[['None',
                                                                                                                                'Po',
                                                                                                                                'Fa',
                                                                                                                                'TA',
                                                                                                                                'Gd',
                                                                                                                                'Ex'],
                                                                                                                               ['None',
                                                                                                                                'Po',
                                                                                                                                'Fa',
                                                                                                                                'TA',
                                                                                                                                'Gd',
                                                                                                                                'Ex'],
                                                                                                                               ['None',
                                                                                                                                'Po',
                                                                                                                                'Fa',
                                                                                                                                'TA',
                                                                                                                                'Gd',
                                                                                                                                'Ex'],
                                                                                                                               ['None'...
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                                   '1stFlrSF', '2ndFlrSF',
                                                   'LowQualFinSF', 'GrLivArea',
                                                   'BsmtFullBath',
                                                   'BsmtHalfBath', 'FullBath',
                                                   'HalfBath', 'BedroomAbvGr',
                                                   'KitchenAbvGr',
                                                   'TotRmsAbvGrd', 'Fir

In [362]:
y_pred = base_pipeline.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [363]:
base_mse = mean_squared_error(y_test,y_pred)
base_rmse = np.sqrt(mean_squared_error(y_test,y_pred))
base_mae = mean_absolute_error(y_test,y_pred)
base_r2 = r2_score(y_test,y_pred)

In [364]:
print('MSE:',base_mse)
print('RMSE:',base_rmse)
print('MAE:',base_mae)
print('R2_SCORE:',base_r2)

MSE: 2008508229.331867
RMSE: 44816.3834923331
MAE: 23244.30371007076
R2_SCORE: 0.7381456570321758


# Baseline Model Results

The baseline model provides the reference performance score for comparison against the feature-engineered model.

# Feature Engineering

Domain-driven features are created based on insights obtained during exploratory data analysis.

Created Features:

- HasGarage
- HasFireplace
- HasPool
- HasBasement
- TotalSF
- HouseAge
- RemodelAge
- TotalBath
- TotalPorchSF

These features aim to capture property characteristics more effectively than raw features alone.

In [365]:
class FeatureCreator(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['HasGarage'] = (X['GarageArea'] > 0).astype(int)
        X['HasFireplaces'] = (X['Fireplaces'] > 0).astype(int)
        X['HasPool'] = (X['PoolArea'] > 0).astype(int)
        X['HasBasement'] = (X['TotalBsmtSF'])
        X['TotalSF'] = X['1stFlrSF']+X['2ndFlrSF']+X['TotalBsmtSF']
        X['HouseAge'] = X['YrSold'] - X['YearBuilt']
        X['RemodelAge'] = X['YrSold'] - X['YearRemodAdd']
        X['TotalBath'] = (X['FullBath'] + 0.5 * X['HalfBath'] + X['BsmtFullBath'] + 0.5 * X['BsmtHalfBath'])
        X['TotalPorchSF'] = (X['OpenPorchSF'] + X['EnclosedPorch'] + X['3SsnPorch'] + X['ScreenPorch'])

        return X

In [366]:
num_cols.extend(['HasGarage','HasFireplaces','HasPool','HasBasement','TotalSF',
                'HouseAge','RemodelAge','TotalBath','TotalPorchSF'])

In [367]:
created_pipeline = Pipeline([
    ('feature_creator',FeatureCreator()),
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
    ])

# Feature Engineered Model

The feature-engineered model uses both original features and newly created features.

The objective is to determine whether feature engineering improves predictive performance.

In [368]:
created_pipeline.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:188: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


Pipeline(steps=[('feature_creator', FeatureCreator()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('ordinal_trf',
                                                  ColumnTransformer(transformers=[('quality',
                                                                                   Pipeline(steps=[('imputer',
                                                                                                    SimpleImputer(fill_value='None',
                                                                                                                  strategy='constant')),
                                                                                                   ('encoder',
                                                                                                    OrdinalEncoder(categories=[['None',
                                                                                                                                'Po',
                                                                                                                                'Fa',
                                                                                                                                'TA',
                                                                                                                                'Gd',
                                                                                                                                'Ex'],
                                                                                                                               ['None',
                                                                                                                                'Po',
                                                                                                                                'Fa',
                                                                                                                                'TA',
                                                                                                                                'Gd',
                                                                                                                                'Ex'],
                                                                                                                               ['No...
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                                   '1stFlrSF', '2ndFlrSF',
                                                   'LowQualFinSF', 'GrLivArea',
                                                   'BsmtFullBath',
                                                   'BsmtHalfBath', 'FullBath',
                                                   'HalfBath', 'BedroomAbvGr',
                                                   'KitchenAbvGr',
                                                   'TotRmsAbvGrd', 'Fireplaces',
                                                   'GarageYrBlt', 'GarageCars',
                                                   'GarageArea', 'WoodDeckSF',
                                                   'OpenPorchSF',
                                                   'EnclosedPorch', ...])])),
                ('model', LinearRegression())])

In [369]:
y_pred = created_pipeline.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [11, 18] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [370]:
created_mse = mean_squared_error(y_test,y_pred)
created_rmse = np.sqrt(mean_squared_error(y_test,y_pred))
created_mae = mean_absolute_error(y_test,y_pred)
created_r2 = r2_score(y_test,y_pred)

In [371]:
print('MSE:',created_mse)
print('RMSE:',created_rmse)
print('MAE:',created_mae)
print('R2_SCORE:',created_r2)

MSE: 1975690440.325864
RMSE: 44448.73946835685
MAE: 23265.505517416703
R2_SCORE: 0.7424241959260305


# Feature Engineered Model Results

The model is evaluated using:

- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- R² Score

# Model Comparison

The baseline and feature-engineered models are compared to evaluate the contribution of engineered features.

Improvement in RMSE, MAE, or R² indicates that the engineered features provide additional predictive information.

In [372]:
comparison = pd.DataFrame({
    'Metric':['RMSE','MAE','R2'],
    'Baseline':[base_rmse,base_mae,base_r2],
    'Feature Engineered':[created_rmse,created_mae,created_r2]
})

comparison

,Metric,Baseline,Feature Engineered
0,RMSE,44816.383492,44448.739468
1,MAE,23244.303710,23265.505517
2,R2,0.738146,0.742424


# Final Conclusions

## Key Achievements

1. Built a complete machine learning pipeline using sklearn Pipeline and ColumnTransformer.

2. Applied specialized preprocessing for:
   - Numerical Features
   - Ordinal Features
   - Nominal Features

3. Implemented domain-driven feature engineering.

4. Compared baseline and feature-engineered models.

5. Evaluated model performance using regression metrics.

## Learning Outcomes

- Advanced Missing Value Handling
- Ordinal Encoding
- One-Hot Encoding
- Power Transformation
- Pipeline Development
- Feature Engineering Evaluation

## Final Outcome

The workflow demonstrates a reusable and production-ready preprocessing pipeline for house price prediction.